In [ ]:
# Cell 8: Rangkuman Metrik & Ekspor Tabel 1
summary_table1 = pd.DataFrame([
    {
        'Radius Buffer': '10 km (Sensitivitas)',
        'Permukiman Terjangkau': reached_10,
        'Persentase': f"{pct_10:.1f}%".replace('.', ',')
    },
    {
        'Radius Buffer': '25 km (Standar Resmi PLN)',
        'Permukiman Terjangkau': reached_25,
        'Persentase': f"{pct_25:.1f}%".replace('.', ',')
    },
    {
        'Radius Buffer': '40 km (Sensitivitas)',
        'Permukiman Terjangkau': reached_40,
        'Persentase': f"{pct_40:.1f}%".replace('.', ',')
    }
])

summary_table1.to_csv("outputs/solarnesia_table1_summary.csv", index=False)
pd.Series(sorted(all_covered_kabs), name="Kabupaten_Kota_Terlayani").to_csv("outputs/solarnesia_covered_kabupaten.csv", index=False)

print("Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:")
print(summary_table1.to_string(index=False))
print("
Files saved: outputs/solarnesia_table1_summary.csv & outputs/solarnesia_covered_kabupaten.csv")


In [ ]:
# Cell 9: Visualisasi Komputer Grafis 2D Statis (Matplotlib)
# Menggabungkan Layer 1 (Choropleth Vektor) dan Layer 2 (Titik Diskrit & Koridor Buffer)
fig = plt.figure(figsize=(16, 9.5), facecolor='#090d16')
ax = fig.add_axes([0.05, 0.18, 0.90, 0.74], facecolor='#090d16')

ax.set_xlim(94.5, 141.5)
ax.set_ylim(-11.5, 6.5)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', color='#1e293b', alpha=0.5, linewidth=0.7)
ax.tick_params(colors='#64748b', labelsize=9)
for spine in ax.spines.values():
    spine.set_color('#1e293b')

# 1. Layer 1: Provincial Base Map & Choropleth
prov_means = gpd.sjoin(land_pts, gdf_prov, how='inner', predicate='intersects').groupby('state')['pvout'].mean()
gdf_prov['mean_pvout'] = gdf_prov['state'].map(prov_means).fillna(land_pts['pvout'].mean())

cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=1250, vmax=1620)
gdf_prov.plot(column='mean_pvout', cmap=cmap, norm=norm, edgecolor='#ffffff', linewidth=0.6, ax=ax, alpha=0.85)

# 2. Layer 2: 25 km Buffer Zones (Translucent Cyan Corridors)
gpd.GeoDataFrame(geometry=plts_3857['buffer_25km'], crs="EPSG:3857").to_crs("EPSG:4326").plot(
    ax=ax, facecolor='#38bdf8', edgecolor='#0284c7', alpha=0.22, linewidth=0.8
)

# 3. Layer 2: Demand Points (Unreached Slate vs Reached Emerald)
demand_4326 = demand_3857.to_crs("EPSG:4326")
demand_4326[~demand_4326['covered_25km']].plot(ax=ax, color='#64748b', markersize=14, alpha=0.55, label='Demand Belum Terjangkau (2.964)')
demand_4326[demand_4326['covered_25km']].plot(ax=ax, color='#10b981', markersize=26, alpha=0.85, edgecolors='#ffffff', linewidth=0.5, label='Demand Terjangkau 25 km (325)')

# 4. Layer 2: PLTS Candidate Sites (Glowing Amber Stars)
plts_candidates.plot(ax=ax, color='#fbbf24', marker='*', markersize=90, edgecolors='#ffffff', linewidth=0.6, label='Kandidat PLTS (513 / 480 Hotspot)')

# Inset Zoom: Jawa Timur, Madura & Bali Detail
ax_ins = fig.add_axes([0.65, 0.55, 0.28, 0.32], facecolor='#0d1527')
ax_ins.set_title("Detail Jangkauan: Jawa Timur, Madura & Bali", color='#f8fafc', fontsize=9.5, fontweight='bold', pad=6)
for s in ax_ins.spines.values():
    s.set_color('#38bdf8')
    s.set_linewidth(1.2)

sub_prov = gdf_prov[gdf_prov['state'].isin(['Jawa Timur', 'Bali', 'Nusa Tenggara Barat'])]
sub_prov.plot(ax=ax_ins, column='mean_pvout', cmap=cmap, norm=norm, edgecolor='#ffffff', linewidth=0.7)

gpd.GeoDataFrame(geometry=plts_3857['buffer_25km'], crs="EPSG:3857").to_crs("EPSG:4326").plot(
    ax=ax_ins, facecolor='#38bdf8', edgecolor='#38bdf8', alpha=0.28, linewidth=1.0
)
demand_4326[demand_4326['covered_25km']].plot(ax=ax_ins, color='#10b981', markersize=35, edgecolors='#ffffff', linewidth=0.6)
demand_4326[~demand_4326['covered_25km']].plot(ax=ax_ins, color='#64748b', markersize=16, alpha=0.6)
plts_candidates.plot(ax=ax_ins, color='#fbbf24', marker='*', markersize=110, edgecolors='#ffffff', linewidth=0.8)

ax_ins.set_xlim(111.0, 116.0)
ax_ins.set_ylim(-9.0, -6.8)
ax_ins.tick_params(colors='#64748b', labelsize=6.5)

# Colorbar & Legend
cbar_ax = fig.add_axes([0.06, 0.08, 0.30, 0.022])
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, orientation='horizontal')
cb.set_label('Rata-rata Potensi Radiasi Surya (PVOUT - kWh/kWp/tahun)', color='#e2e8f0', fontsize=8)
cb.ax.tick_params(labelsize=7.5, colors='#94a3b8')

leg = ax.legend(loc='lower right', facecolor='#0f172a', edgecolor='#334155', fontsize=8.5)
for t in leg.get_texts():
    t.set_color('#f1f5f9')

fig.text(0.05, 0.94, "SOLARNESIA: MODEL INTEGRASI SPASIAL DUA LAPIS PLTS NASIONAL", fontsize=16, fontweight='bold', color='#f8fafc')
fig.text(0.05, 0.91, "Layer 1: Choropleth Poligon Daratan | Layer 2: 513 Titik PLTS (480 Hotspot) & Koridor Feeder 25 km PLN 20 kV", fontsize=10, color='#38bdf8')

out_png = "outputs/solarnesia_coverage_map.png"
fig.savefig(out_png, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f"Composite 2D visualization exported to: {out_png}")


In [ ]:
# Cell 10: Peta Spasial Interaktif & Inspeksi Klik (Folium)
# Klik pada sembarang titik (misal Bali Selatan) untuk melihat nama-nama kecamatan/kabupaten dalam radius 25 km
print("Generating interactive Folium web map...")
# Center map on Indonesia (Bali & Java focal area)
center_lat, center_lon = -8.40, 115.20
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles="OpenStreetMap",
    control_scale=True
)

# 1. Add 25 km Buffer Polygons (Translucent Cyan)
buffer_layer = folium.FeatureGroup(name="Koridor Distribusi 25 km (PLN 20 kV)")
for idx, row in plts_candidates.iterrows():
    buf_circle = folium.Circle(
        location=[row['latitude'], row['longitude']],
        radius=25000,
        color="#38bdf8",
        weight=1.5,
        fill=True,
        fill_color="#38bdf8",
        fill_opacity=0.15,
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Radius:</b> 25 km"
    )
    buf_circle.add_to(buffer_layer)
buffer_layer.add_to(m)

# 2. Add Reached Demand Points (Emerald Dots)
demand_layer = folium.FeatureGroup(name="Permukiman Terjangkau 25 km (325 Titik)")
for idx, row in demand_4326[demand_4326['covered_25km']].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3.5,
        color="#10b981",
        fill=True,
        fill_color="#10b981",
        fill_opacity=0.85,
        tooltip=f"Demand ID: {idx} | Terjangkau"
    ).add_to(demand_layer)
demand_layer.add_to(m)

# 3. Add PLTS Candidate Markers with Click Popups
plts_layer = folium.FeatureGroup(name="Kandidat PLTS & Hotspot Sentra (513 Titik)")
for idx, row in plts_candidates.iterrows():
    popup_html = f"""
    <div style='font-family: Arial, sans-serif; font-size: 12px; width: 260px; line-height: 1.5;'>
        <b style='color: #d97706; font-size: 13px;'>☀️ {row['hotspot_id']}</b><br>
        <b>Koordinat:</b> {row['latitude']:.4f}° LS, {row['longitude']:.4f}° BT<br>
        <b>Potensi PVOUT:</b> <span style='color: #059669; font-weight: bold;'>{row['pvout']:.1f} kWh/kWp</span><br>
        <b>Suhu Rata-rata:</b> {row['temp']:.1f} °C<br>
        <hr style='margin: 6px 0; border: 0; border-top: 1px solid #e5e7eb;'>
        <b>Wilayah Terjangkau (Radius 25 km):</b><br>
        <span style='color: #1e3a8a; font-size: 11px;'>{row['covered_areas_25km']}</span>
    </div>
    """
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color="#ffffff",
        weight=1.5,
        fill=True,
        fill_color="#f59e0b",
        fill_opacity=0.95,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"<b>Klik untuk Inspeksi:</b> {row['hotspot_id']} (PVOUT: {row['pvout']:.0f})"
    ).add_to(plts_layer)
plts_layer.add_to(m)

folium.LayerControl().add_to(m)
out_html = "outputs/solarnesia_interactive_map.html"
m.save(out_html)
print(f"Interactive web map saved to: {out_html}")
print("Click on any amber PLTS marker (e.g. Southern Bali) to inspect the 25km covered areas.")
m
